<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Archive/targetgroup_targets_landuses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libaries

In [65]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb

In [66]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [67]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [68]:
# Fetch data
tabel0 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets
tabel2 = fetch_airtable_data("tblTRyuT48bBN24QG")  # Land Uses

In [78]:
tabel1.columns.tolist()

['Target name',
 'Policy Source',
 'Quotes (text excerpts with references)',
 'Description of target',
 'Target Group',
 'terest',
 'Created',
 'id',
 'Target time frame',
 'Functions',
 'Land uses',
 'Land conditions']

In [95]:
# Rename and explode
# Target Groups
target_groups = tabel0.copy()
target_groups = target_groups.explode("Targets").rename(columns={
    "Target Group": "target_group_id",
    "Targets": "target_id"
})

targets = tabel1.copy()
targets = targets.rename(columns={
    "id": "target_id",
    "Target name": "target_name"  # så du stadig har navnet til labels
})

# Land Uses
land_uses = tabel2.copy()
land_uses = land_uses.explode("Targets").rename(columns={
    "Name": "land_uses_id",
    "Targets": "target_id"
})


In [96]:
merged_1 = pd.merge(targets, target_groups, on="target_id", how="left")
merged_2 = pd.merge(merged_1, land_uses, on="target_id", how="left")

#Test
merged_2[['target_group_id', 'target_id', 'land_uses_id']].dropna().head()



,target_group_id,target_id,land_uses_id
3,Climate,rec0JfZPYjBdr4YTr,"Genopretning af heder, enge, ådale og moser"
4,Climate,rec0JfZPYjBdr4YTr,Kulstofoptag- og binding i skove og jorde
5,Climate,rec0JfZPYjBdr4YTr,Vådområder
6,Climate,rec0JfZPYjBdr4YTr,Beskyttede naturområder
7,Climate,rec0JfZPYjBdr4YTr,Ekstensivt dyrkede græsarealer


In [97]:
import hashlib

def label_to_rgba(label, alpha=0.4):
    if isinstance(label, str):
        hash_object = hashlib.md5(label.encode())
        hash_hex = hash_object.hexdigest()
        r = int(hash_hex[0:2], 16)
        g = int(hash_hex[2:4], 16)
        b = int(hash_hex[4:6], 16)
        return f'rgba({r},{g},{b},{alpha})'
    else:
        return f'rgba(200,200,200,{alpha})'

def label_to_id(label, label_list):
    if label not in label_list:
        label_list.append(label)
    return label_list.index(label)


In [98]:
all_labels = []
links = []

for _, row in merged_2.dropna(subset=["target_group_id", "target_id", "land_uses_id"]).iterrows():
    # Første led: Target Group → Target
    source1 = label_to_id(row["target_group_id"], all_labels)
    target1 = label_to_id(row["target_id"], all_labels)
    color1 = label_to_rgba(row["target_group_id"])

    links.append({
        "source": source1,
        "target": target1,
        "value": 1,
        "color": color1
    })

    # Andet led: Target → Land Use
    target2 = label_to_id(row["land_uses_id"], all_labels)
    color2 = label_to_rgba(row["target_id"])

    links.append({
        "source": target1,
        "target": target2,
        "value": 1,
        "color": color2
    })


In [99]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="gray", width=0.5),
        label=all_labels,
        color=[label_to_rgba(label, alpha=1.0) for label in all_labels]
    ),
    link=dict(
        source=[link["source"] for link in links],
        target=[link["target"] for link in links],
        value=[link["value"] for link in links],
        color=[link["color"] for link in links]
    )
)])

fig.update_layout(title_text="Target Group → Target → Land Use", font_size=12, height=1000)
fig.show()


In [94]:
print("🔹 Eksempler fra targets:")
print(targets["target_id"].dropna().head())

print("\n🔹 Eksempler fra land_uses:")
print(land_uses["target_id"].dropna().head())


🔹 Eksempler fra targets:
0    Beskyttelse borgernes sundhed og trivsel mod m...
1    God økologisk tilstand for vandmiljøet gennem ...
2    Beskyttelse af primære og gamle skovøkosysteme...
3    Beskyttelse af drikkevandet gennem stop i anve...
4    Øget plads til andre formål  såsom fx biomasse...
Name: target_id, dtype: object

🔹 Eksempler fra land_uses:
0    rec2IIEoRzoDUSppM
2    reccUNzKYSD5rHXaI
2    recPTNg97J0dQl8Sf
2    recItPZ5hoXek2iEH
3    recXuUD4EgCZgttzE
Name: target_id, dtype: object


Gl kode

In [ ]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3", tabel3)

In [ ]:
# SQL query
query = """
SELECT
    t0."Policy source"       AS policy_source,
    t1."Target name"         AS mid_target,
    t2."Target Group"        AS target_group,
    t3."Target name"         AS final_target
FROM
    tabel0 t0
JOIN
    tabel1_exp t1 ON t0.target_id = t1.id
JOIN
    tabel2_exp t2 ON t1.target_group_id = t2.id
JOIN
    tabel3 t3 ON t2.target_id = t3.id
"""

results = duckdb.sql(query).df()

In [ ]:
# Trin 1: forkort og saml labels
results['final_target'] = results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels = results['policy_source']
middle_labels = results['target_group']
target_labels = results['final_target']
all_labels = pd.concat([source_labels, middle_labels, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

In [ ]:
# Trin 2: forbindelser
links1 = pd.DataFrame({
    'source': source_labels.map(label_to_index),
    'target': middle_labels.map(label_to_index),
    'value': 1
})
links2 = pd.DataFrame({
    'source': middle_labels.map(label_to_index),
    'target': target_labels.map(label_to_index),
    'value': 1
})
all_links = pd.concat([links1, links2])

In [ ]:
# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color="lightgray"
    )
)])
fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=2000)
fig.show()

#Selektering grøn trepart

In [ ]:
# prompt: make a sankey diagram with only Aftale om et grønt Danmark (grøn trepart) as policy source

import pandas as pd
# Selektering grøn trepart
filtered_results = results[results['policy_source'] == 'Aftale om et grønt Danmark (grøn trepart)'].copy()

# Trin 1: forkort og saml labels
filtered_results['final_target'] = filtered_results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels_filtered = filtered_results['policy_source']
middle_labels_filtered = filtered_results['target_group']
target_labels_filtered = filtered_results['final_target']

# Only include labels that are present in the filtered data
all_labels_filtered = pd.concat([source_labels_filtered, middle_labels_filtered, target_labels_filtered])
unique_labels_filtered = pd.unique(all_labels_filtered)
label_to_index_filtered = {label: i for i, label in enumerate(unique_labels_filtered)}

# Trin 2: forbindelser for filtreret data
links1_filtered = pd.DataFrame({
    'source': source_labels_filtered.map(label_to_index_filtered),
    'target': middle_labels_filtered.map(label_to_index_filtered),
    'value': 1
})

links2_filtered = pd.DataFrame({
    'source': middle_labels_filtered.map(label_to_index_filtered),
    'target': target_labels_filtered.map(label_to_index_filtered),
    'value': 1
})

all_links_filtered = pd.concat([links1_filtered, links2_filtered])

# Remove any links where the source or target index is NaN (due to filtering)
all_links_filtered.dropna(subset=['source', 'target'], inplace=True)
all_links_filtered['source'] = all_links_filtered['source'].astype(int)
all_links_filtered['target'] = all_links_filtered['target'].astype(int)


# Trin 3: Sankey-diagram for filtreret data
fig_filtered = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels_filtered),
    ),
    link=dict(
        source=all_links_filtered['source'],
        target=all_links_filtered['target'],
        value=all_links_filtered['value'],
        color="lightgray"
    )
)])

fig_filtered.update_layout(title_text="Aftale om et grønt Danmark (grøn trepart) → Target Group → Target", font_size=12, height=2500)
fig_filtered.show()

In [ ]:
# prompt: download an interactive html file

fig.write_html("sankey_diagram.html")
from google.colab import files
files.download("sankey_diagram.html")

fig_filtered.write_html("sankey_diagram_filtered.html")
files.download("sankey_diagram_filtered.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>